In [ ]:
import openai
from bertopic.representation import OpenAI
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords 
import re
import json

# Dataset
DATA_PATH = r"C:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\Common\NTSB_ALL.csv"
GLOSSARY_PATH = (
    r"C:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\Common\Glossary.json"
)

NTSB_df = pd.read_csv(DATA_PATH)
# Define regex pattern to remove URLs, "NTSB", and "The Safety Board"
pattern = r"(?:https?://\S+|www\.\S+)|NTSB|The Safety Board(?:'s)?"

# Apply cleaning to narrative column
NTSB_df['narrative'] = NTSB_df['narrative'].apply(
    lambda text: re.sub(pattern, '', text, flags=re.IGNORECASE).strip()
)

with open(GLOSSARY_PATH, "r", encoding="utf-8") as f:
    ABBREVIATIONS = json.load(f)


def expand_abbreviations(text, abbr_dict):
    if pd.isna(text):
        return text

    text = str(text)

    for abbr, full in abbr_dict.items():
        pattern = r"\b" + re.escape(abbr) + r"\b"
        text = re.sub(pattern, full, text, flags=re.IGNORECASE)

    return text

NTSB_df["narrative"] = (
    NTSB_df["narrative"]
    .astype(str)
    .apply(lambda x: expand_abbreviations(x, ABBREVIATIONS))
)


stopwords = list(stopwords.words('english'))
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words="english")

umap_model = UMAP(n_neighbors=5, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=25, min_samples=8, prediction_data=True, gen_min_span_tree=True, metric='euclidean', cluster_selection_method='eom')

client = openai.OpenAI(
    base_url='http://localhost:11434/v1',  # URL where Ollama is running
    api_key='ollama',  # Required but unused
)

# KeyBERT
keybert = KeyBERTInspired()

# MMR
mmr = MaximalMarginalRelevance(diversity=0.3)

system_prompt = """
<s>[INST] <<SYS>>
You are a helpful, respectful, concise and honest assistant for labeling topics in aviation accidents.
<</SYS>>
"""

# Example prompt demonstrating the output we are looking for
example_prompt = """
I have a topic that contains the following documents, each describing an aircraft accident:
- During cruise, the aircraft experienced a sudden loss of engine power. Investigation revealed damage to the turbine blades.
- A fire broke out in the engine compartment shortly after takeoff, leading to an emergency landing.
- Engine inspection after flight revealed fuel contamination and damaged components causing power loss.

The topic is described by the following keywords: 'engine, fuel, blade, examination, fire, power, revealed'.

Based on the information about the topic above, please create a **short label (1-3 words)** of this topic that represents the main cause of the accidents. Make sure to only return the label and nothing more.

[/INST] Engine failure
"""

# Our main prompt with documents ([DOCUMENTS]) and keywords ([KEYWORDS]) tags
main_prompt = """
[INST]
I have a topic that contains the following documents, each describing an aircraft accident:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information about the topic above, please create a **short label (1-3 words)** of this topic that represents the main cause of the accidents. Make sure you to only return the label and nothing more.
[/INST]
"""

prompt = system_prompt + example_prompt + main_prompt
# OpenAI Text Generation
openai = OpenAI(client, model='llama3.1:latest',prompt=prompt, max_length=100, temperature=0.1)

# Create your representation model
representation_model = { "OpenAI": openai, "KeyBERT": keybert, "MMR": mmr }

# Create your BERTopic model
model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    representation_model=representation_model,
    language="english",
    calculate_probabilities=True,
    verbose=True
)

topics, probs = model.fit_transform(NTSB_df['narrative'].tolist())
model.get_topic_info() 

In [ ]:
llama2_labels = [label[0][0].split("\n")[0] for label in model.get_topics(full=True)["OpenAI"].values()]
model.set_topic_labels(llama2_labels)
model.visualize_documents(NTSB_df['narrative'].tolist(), hide_annotations=True, hide_document_hover=False, custom_labels=True)

In [ ]:
# Apply FCI
llama2_labels = [label[0][0].split("\n")[0] for label in model.get_topics(full=True)["OpenAI"].values()]
model.set_topic_labels(llama2_labels)
# --- Binarization of matrix ---
binary_matrix = (probs >= 0.1).astype(int)
# substitute headers with topic names
llama2_34labels = llama2_labels[1:]  # Exclude the -1 label if present
binary_df = pd.DataFrame(binary_matrix, columns=[f"{label}" for i, label in enumerate(llama2_34labels)])


from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.GraphUtils import GraphUtils
import matplotlib.pyplot as plt
import pandas as pd

## --- Run FCI algorithm ---
# Default parameters: fisherz test, alpha=0.05, depth=-1 (unlimited), max_path_length=-1 (unlimited)
g, edges = fci(binary_df.values, alpha=0.05, independence_test_method="chisq")

# --- Convert to pydot for visualization ---
pydot_graph = GraphUtils.to_pydot(g, labels=llama2_34labels)
pydot_graph.write_png("causal_topics_labeled_fci.png")

# --- Display the image ---
plt.figure(figsize=(12,12))
plt.imshow(plt.imread("causal_topics_labeled_fci.png"))
plt.axis("off")
plt.title("Causal Relationships Between Topics (FCI)")
plt.show()

In [ ]:
binary_df.to_csv("NTSB_topics_binary_matrix.csv", index=False)
probs_df = pd.DataFrame(probs, columns=[f"Topic_{i}" for i in range(probs.shape[1])])
probs_df.to_csv("NTSB_topics_probabilities.csv", index=False)